# Required vs Optional Fields

So far, all the fields we defined in our Pydantic model were required.

As an analogy, how do we make function arguments optional in Python?

We provide the argument a default value.

The same approach is used by Pydantic.

We make a field optional, by simply providing the field definition with a default value.

There are a few ways of doing this, we'll explore one way here - and explore other ways later in the course.

In [18]:
from pydantic import BaseModel

class Circle(BaseModel):
    center: tuple[int, int] = (0, 0)
    radius: int

In this model, `center` is an **optional** field, that will default to `(0, 0)` if not provided in the data we are deserializing. on the other hand, since `radius` does not have a default defined, it is a **required** field.

We can also see this by inspecting the model fields:

In [19]:
Circle.model_fields

{'center': FieldInfo(annotation=tuple[int, int], required=False, default=(0, 0)),
 'radius': FieldInfo(annotation=int, required=True)}

Note how `center` has the `required=False` property, while `radius` has `required=True`.

We can now create instances of `Circle` without providing a value for `center`:

In [20]:
Circle(radius=1)

Circle(center=(0, 0), radius=1)

As you can see, no validation exception was raised, and we have the proper default value in place.

This works the same way for all the other deserialization methods too:

In [ ]:
data = {"radius": 1}
data_json = '{"radius": 1}'

In [21]:
Circle.model_validate(data)

Circle(center=(0, 0), radius=1)

In [22]:
Circle.model_validate_json(data_json)

Circle(center=(0, 0), radius=1)

Of course, we can provide a value for center too:

In [23]:
Circle(center=(1, 1), radius=2)

Circle(center=(1, 1), radius=2)

We have to be quite careful about one thing when specifying a default value for a field.

When we provide a field value, Pydantic validates that value before putting it into our model instance. However, Pydantic's default behavior does not validate default values!

That kind of makes sense - after all, we are writing the model definition, so we should be able to only provide a valid default value in our model definition.

So, here's the issue:

In [24]:
class Model(BaseModel):
    field: int = "Python"

As you can see, the provided default is totally inconsistent for an `int` type, and yet this still works:

In [25]:
Model()

Model(field='Python')

So, be careful when providing default values, the onus is on us, as developers, to make sure we provided consistent defaults.

Pydantic does offer us a way to force default data validations - but it does mean a little of extra compute time to validate that default value every time it is needed. Something we can avoid if we write correct code. We'll see later how to enable default validations on a model.

Usually in Python, including dataclasses, we have to be extra careful when we assign a default that is a mutable object.

let's look at an example of this with a regular Python function:

By the way, writing a function that behaves this way - modifies it's input and returns it, is a terrible coding technique - I'm just using this to illustrate a point.

We can call this function with our own list:

In [27]:
from time import time

def extend_list(user_list: list = []):
    user_list.append(int(time()))
    return user_list

In [28]:
my_times = []
extend_list(my_times)
my_times

[1776358979]

In [29]:
my_times = extend_list()
my_times

[1776358997]

And this seems to have worked.

But what about this?

In [30]:
my_new_times = extend_list()

Our expectation might be that `my_new_times` just contains one element, whatever the epoch time was when we called `extend_list()`.

In [31]:
my_new_times

[1776358997, 1776359013]

Huh??

This is the issue I was telling you about. When we defined a default for function arguments, these values are calculated and stored with the function itself - they are not re-created every time the function is called. In other words, that default value is going to be **shared** amongst all the function calls.
<div style="
    direction: rtl;
    background:  #506666;
    padding: 40px 30px;
    border-radius: 14px;
    font-family: 'Vazirmatn', Tahoma;
    line-height: 2;
    color: #fcfcfc;
    display: flex;
    align-items: center;
    gap: 12px;
    font-size: 20px;
    width: 85%; margin: 10px auto;
">

<span style="font-size: 38px; margin-right:25px">🔍</span>

در پایتون، وقتی شما با کلمه کلیدی def یک تابع رو تعریف می‌کنی، پایتون فقط و فقط 1بار همون لحظه‌ی تعریف، مقادیر پیش‌فرض رو تو حافظه می‌سازه و ارزیابی می‌کنه.
تو کد شما، مقدار <span dir="ltr"><code>user_list: list = []</code></span>
 در زمان اجرای اولیه اسکریپت ساخته میشه، نه زمانی که تابع رو صدا می‌زنی!
یعنی این لیست [] یک جایگاه ثابت تو حافظه می‌گیره. حالا اگر تابع رو بدون آرگومان صدا بزنی، پایتون هر بار همون لیست قدیمی و ثابت رو بهت میده، نه یک لیست خالی جدید!

</div>

<div style="
    direction: rtl;
    background:  #1f3131;
    padding: 40px 30px;
    border-radius: 14px;
    font-family: 'Vazirmatn', Tahoma;
    line-height: 2;
    color: #fcfcfc;
    display: flex;
    align-items: center;
    gap: 12px;
    font-size: 20px;
    width: 85%; margin: 10px auto;
">

<span style="font-size: 38px; margin-right:25px">🔍</span>

بچه‌ها! لیست‌ها و دیکشنری‌ها تو پایتون مثل یک ظرف غذا می‌مونن. وقتی ظرف رو به عنوان پیش‌فرضِ تابع قرار می‌دین، پایتون فقط یک بار اون ظرف رو می‌سازه! اگه تو اجرای اول تابع توش قرمه‌سبزی بریزید، تو اجرای دوم همون ظرف قرمه‌سبزی دار میاد جلوتون! پس حتماً برای نوع‌های داده تغییرپذیر (Mutable) از مقدار None استفاده کنید یا تو Pydantic از default_factory کمک بگیرید.”Shared Memory


</div>

Something similar happens with dataclasses. In dataclasses, we get around the problem by a mechanism that is called a default factory. Instead of providing a mutable object as a default value, we provide a function that will get called each time a dataclass instance is created, and that function will therefore return a **new** default object.

<div style="
    direction: rtl;
    background:  #2b4141;
    padding: 40px 30px;
    border-radius: 14px;
    font-family: 'Vazirmatn', Tahoma;
    line-height: 2;
    color: #fcfcfc;
    display: flex;
    align-items: center;
    gap: 12px;
    font-size: 20px;
    width: 85%; margin: 10px auto;
">

<span style="font-size: 38px; margin-right:25px">🔍</span>

متن داره میگه: مشکلاتی که تو توابع داشتیم (یعنی همون به اشتراک گذاشته شدن یک لیست بین همه)، تو Dataclassها هم وجود داره. اما پایتون تو Dataclass اومده یک مکانیزم هوشمندانه به اسم Default Factory (کارخانه پیش‌فرض) معرفی کرده.

به جای اینکه بیایم خود شیء (مثلاً یک لیست خالی []) رو همون اول بدیم، میایم دستورالعمل ساختنش (یک تابع) رو میدیم! اینطوری هر بار که یک نمونه جدید از کلاس ساخته میشه، اون تابع (کارخانه) یک بار اجرا میشه و یک شیء کاملاً جدید و مستقل (New default object) تولید می‌کنه.
</div>



Pydantic is very similar, in that it offers this default factory idea (which we'll cover later).

However, it goes one step further, and actually allows us to simply define mutable objects as defaults. 

When Pydantic sees this, it will actually create a deep copy of the mutable object every time a new model instance is created.

In [32]:
print(extend_list()) # خروجی: [1713260000]
print(extend_list()) # خروجی: [1713260000, 1713260005] 
print(extend_list()) # خروجی: [1713260000, 1713260005, 1713260010]


[1776358997, 1776359013, 1776359857]
[1776358997, 1776359013, 1776359857, 1776359857]
[1776358997, 1776359013, 1776359857, 1776359857, 1776359857]


<div dir="rtl" align="right">

> 👋 **! به عنوان توسعه‌دهنده ارشدت باید بگویم...**
> 
> ⚠️ این متن به یکی از معروف‌ترین و خطرناک‌ترین تله‌های پایتون (**Python Gotchas**) اشاره می‌کند که به آن **آرگومان‌های پیش‌فرض تغییرپذیر** (Mutable Default Arguments) 🪤 می‌گویند.
> 
> 🚀  دقیقاً مرز و تفاوت بین یک برنامه‌نویس مبتدی 👶 و یک توسعه‌دهنده حرفه‌ای 🥷 را نشان می‌دهد!
> 
> 💻 بیایید این سناریو را به زبان ساده و با مثال‌های واقعی برنامه‌نویسی (در سطح Developer 🛠️) با هم تحلیل کنیم...

</div>


<div dir="rtl" align="right" style="width: 85%; margin: 10px auto; background-color: #fff3cd; color: #856404; padding: 15px; border-radius: 8px; border-right: 5px solid #ffeeba;">

<span style="font-size: 1.2em;">این متن در واقع دارد <b>سیر تکامل حل یک مشکل بزرگ</b> در پایتون را توضیح می‌دهد. من آن را به ۳ بخش تقسیم می‌کنم:</span>
<br><br>

<b>🚨 ۱. مشکل پایتون خالص (The Python Trap)</b><br>
متن می‌گوید وقتی در پایتون یک مقدار پیش‌فرضِ تغییرپذیر (مثل لیست <code>[]</code> یا دیکشنری <code>{}</code>) به یک تابع می‌دهیم، پایتون آن را <b>فقط یک‌بار</b> زمان تعریف تابع می‌سازد، نه هر بار که تابع صدا زده می‌شود!

</div>


In [33]:
# فاجعه در پایتون خالص
def add_item(item, my_list=[]):
    my_list.append(item)
    return my_list

print(add_item("A")) # خروجی: ['A']
print(add_item("B")) # خروجی: ['A', 'B']  <-- فاجعه! لیست بین فراخوانی‌ها به اشتراک گذاشته شد!


['A']
['A', 'B']


<div dir="rtl" align="right" style=" width: 85%; margin: 10px auto; background-color: #d1ecf1; color: #0c5460; padding: 15px; border-radius: 8px; border-right: 5px solid #117a8b;">

<b>💡 ۲. راه حل Dataclasses (استفاده از default_factory)</b><br><br>
متن اشاره می‌کند که <code>dataclasses</code> در پایتون برای حل این مشکل، مفهومی به نام <code>default_factory</code> معرفی کرد. یعنی به جای اینکه بگوییم پیش‌فرضِ تو لیستِ خالی <code>[]</code> است، یک <b>کارخانه یا تابع</b> (مثل <code>list</code>) به آن می‌دهیم تا هر بار یک شیء جدید بسازد. 🏭

</div>


In [ ]:
from dataclasses import dataclass, field

@dataclass
class User:
    tags: list = [] # این خطا می‌دهد!
    tags: list = field(default_factory=list) # راه حل اصولی


<div dir="rtl" align="right" style="  width: 85%; margin: 10px auto;background-color: #1b1b1a;15px; border-radius: 8px; border-right: 5px solid #9333ea;">

<b>🪄 ۳. جادوی Pydantic (تکامل نهایی)</b><br><br>
در پاراگراف آخر، متن شاهکار <b>Pydantic</b> (که قلب تپنده <code>FastAPI</code> است) را رو می‌کند. پایدانتیک هم <code>default_factory</code> را دارد، اما یک قدم فراتر می‌رود: حتی اگر به اشتباه از <code>[]</code> استفاده کنید، پایدانتیک پشت صحنه یک <b>Deep Copy (کپی عمیق)</b> از آن می‌گیرد تا از به اشتراک‌گذاری حافظه جلوگیری کند! سپر دفاعی پایدانتیک اینجاست! 🛡️

</div>


<div dir="rtl" align="right" style="background-color: #202220;width: 85%; margin: 10px auto;; border-radius: 8px; border-right: 5px solid #28a745;">

<b>🧙‍♂️ روش اول: جادوی پنهان Pydantic (استفاده مستقیم از لیست خالی)</b><br><br>
در این مثال نشان می‌دهیم که اگر برنامه‌نویس اشتباه کند و مثل پایتون خالص از <code>[]</code> استفاده کند، <b>Pydantic</b> چطور با یک کپی عمیق (<b>Deep Copy</b>) جلوی فاجعه را می‌گیرد و از دیتای ما محافظت می‌کند. 🛡️

</div>


In [34]:
from pydantic import BaseModel

class UserMagic(BaseModel):
    username: str
    # در پایتون خالص این یک فاجعه است، اما Pydantic آن را مدیریت می‌کند
    tags: list[str] = []

# کاربر اول را می‌سازیم و یک تگ به او می‌دهیم
user1 = UserMagic(username="ali_dev")
user1.tags.append("fastapi")

# کاربر دوم را می‌سازیم بدون اینکه تگی بدهیم
user2 = UserMagic(username="reza_ops")

print(f"User 1 tags: {user1.tags}") # خروجی: ['fastapi']
print(f"User 2 tags: {user2.tags}") # خروجی: [] 

# بررسی آدرس در حافظه (اینجا جادو ثابت می‌شود)
# $id(user1.tags) \neq id(user2.tags)$


User 1 tags: ['fastapi']
User 2 tags: []


<div dir="rtl" align="right" style="width: 85%; margin: 10px auto; background-color: #fff8dc; color: #8b6508; padding: 20px; border-radius: 8px; border-right: 5px solid #daa520; line-height: 1.8;">

<b>👨‍🏫 نکته طلایی :</b><br>
<code>Pydantic</code> پشت صحنه تشخیص می‌دهد که <code>[]</code> یک شیء تغییرپذیر (Mutable) است و برای هر نمونه (Instance) جدید، یک فضای حافظه جدید ($Memory \ Address$) اختصاص می‌دهد. 🧠

</div>


<div dir="rtl" align="right" style="width: 85%; margin: 10px auto; background-color: #fff8dc; color: #8b6508; padding: 20px; border-radius: 8px; border-right: 5px solid #daa520; line-height: 1.8;">

<b>👨‍🏫روش دوم: روش استاندارد و Best Practice (استفاده از default_factory)

به عنوان یک برنامه‌نویس حرفه‌ای، بهتر است همیشه کدمان خوانا (Explicit) باشد و به جادوی فریم‌ورک‌ها تکیه نکنیم. روش استاندارد در Pydantic (و فیلدهای اختیاری پیچیده در FastAPI) استفاده از Field و default_factory است🧠

</div>


In [35]:
from pydantic import BaseModel, Field

class UserStandard(BaseModel):
    username: str
    # روش اصولی و حرفه‌ای: استفاده از default_factory
    tags: list[str] = Field(default_factory=list)

user_a = UserStandard(username="sara_ai")
user_a.tags.append("machine_learning")

user_b = UserStandard(username="mina_web")

print(f"User A tags: {user_a.tags}") # خروجی: ['machine_learning']
print(f"User B tags: {user_b.tags}") # خروجی: []


User A tags: ['machine_learning']
User B tags: []
